# **다중분류 실습: MNIST 손글씨 분류**

<center><img src = "https://github.com/Jangrae/img/blob/master/mnist.png?raw=true" width=800/></center>

- MNIST는 28×28 픽셀 흑백 손글씨 숫자(0~9) 이미지 데이터 셋입니다.
- 구성은 학습용 60,000장, 검용 10,000장으로 되어 있습니다.
- 머신러닝, 딥러닝 모델 학습과 기본 성능 평가에 사용됩니다.

## **1. 환경준비**

### (1) 라이브러리 불러오기

- 학습량이 많기 때문에 GPU를 사용하는 것이 좋습니다.

In [ ]:
# GPU 사용 가능 확인
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print('* GPU 사용 가능:', gpus[0])
else:
    print('* GPU 사용 불가')

In [ ]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.backend import clear_session
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import mnist
import warnings

warnings.filterwarnings(action='ignore')
%config InlineBackend.figure_format = 'retina'

### (2) 함수 만들기

In [ ]:
# 함수 만들기
def dl_history_plot(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history['loss'], label='Train Loss', marker='.')
    plt.plot(history['val_loss'], label='Validation Loss', marker='.')

    plt.title('Learning Curve', size=15, pad=20)
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend()
    plt.show()

### (3) 데이터 준비

In [ ]:
# MNIST 불러오기
(x_train, y_train), (x_val, y_val) = mnist.load_data()

In [ ]:
# 크기 확인
print(x_train.shape)
print(y_train.shape)
print('-' * 15)
print(x_val.shape)
print(y_val.shape)

## **2. 데이터 살펴보기**

In [ ]:
# 아래 숫자를 바꿔가며 확인
n = 20
plt.figure()
plt.imshow(x_train[n], cmap=plt.cm.binary)
plt.colorbar()
plt.show()

In [ ]:
# 값 확인
n = 20
for i in x_train[n]:
    for j in i:
        print(f'{j:3d}', end='')
    print('')

In [ ]:
# 25개 확인
class_names = list('0123456789')
plt.figure(figsize=(6, 6))
for i in range(25):
    plt.subplot(5, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(x_train[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[y_train[i]])
plt.tight_layout()
plt.show()

## **3. 데이터 전처리**

### (1) 2차원으로 펼치기

In [ ]:
# 변환 전
print(x_train.shape, y_train.shape)
print(x_val.shape, y_val.shape)

In [ ]:
# 2차원 변환
x_train = x_train.reshape(60000, -1)
x_val = x_val.reshape(10000, -1)

In [ ]:
# 변환 후
print(x_train.shape, y_train.shape)
print(x_val.shape, y_val.shape)

### (2) 스케일링

- 0 - 255 값으로 되어 있는 데이터를 0 - 1사이 값으로 변환합니다.
- x_train, x_val을 그냥 255로 나누면 됩니다.

In [ ]:
# 0 ~ 1 사이 값으로 변환
x_train = x_train / 255
x_val = x_val / 255

## **4. 모델링 1**

### (1) 모델 선언

In [ ]:
# 메모리 정리
clear_session()

# 입력 Feature 수
nfeatures = x_train.shape[1]

# Sequential 모델 선언
model = Sequential([
    Input(shape=(nfeatures, )),
    Dense(10, activation='softmax')
])

# 모델 요약
model.summary()

### (2) 모델 학습

In [ ]:
# 학습 설정
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy')

In [ ]:
# 모델 학습
hist = model.fit(x_train, y_train, batch_size=128, epochs=20, validation_split=0.2, verbose=0).history

In [ ]:
# 학습 곡선
dl_history_plot(hist)

### (3) 예측 및 성능 평가

In [ ]:
# 예측
y_pred = model.predict(x_val)
y_pred = np.argmax(y_pred, axis=1)

In [ ]:
# 성능 평가
print(confusion_matrix(y_val, y_pred))
print('-' * 53)
print(classification_report(y_val, y_pred))

## **5. 모델링 2**

- Hidden Layer를 추가하여 다양한 모델을 만들고 성능을 비교해 봅니다.
- 성능에 영향을 주는 요인은
    - Hidden Layer 수
    - Node 수
    - epochs 수 (10 ~ 20 사이에서 결정)
    - learning_rate

#### 1) 모델 선언

In [ ]:
# 메모리 정리
clear_session()

# 입력 Feature 수
nfeatures = x_train.shape[1]

# Sequential 모델 선언
model = Sequential([
    Input(shape=(nfeatures, )),
    Dense(392, activation='relu'),
    Dense(196, activation='relu'),
    Dense(98, activation='relu'),
    Dense(10, activation='softmax')
])

# 모델 요약
model.summary()

#### 2) 모델 학습

In [ ]:
# 학습 설정
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy')

In [ ]:
# 모델 학습
hist = model.fit(x_train, y_train, batch_size=128, epochs=20, validation_split=0.2, verbose=0).history

In [ ]:
# 학습 곡선
dl_history_plot(hist)

#### 3) 예측 및 성능 평가

In [ ]:
# 예측
y_pred = model.predict(x_val)
y_pred = np.argmax(y_pred, axis=1)

In [ ]:
# 성능 평가
print(confusion_matrix(y_val, y_pred))
print('-' * 53)
print(classification_report(y_val, y_pred))